# Lecture 4 self-check: Debugging and testing: answers

The answers to the Lecture 4 self-check, with a short explanation for each, including why the tempting wrong options are wrong. You can check most of them by running the code yourself.

### Question 1

What does running this line raise?

```python
from math import sinn
```

- **(A)** `AttributeError`
- **(B)** `NameError`
- **(C)** `ImportError`
- **(D)** `ModuleNotFoundError`

**Answer: (C)**

The module `math` exists, so Python imports it, but it has no name `sinn` (a typo for `sin`). Importing a name that a module doesn't contain raises `ImportError` ("cannot import name 'sinn' from 'math'"). `ModuleNotFoundError`, a subclass of `ImportError`, is raised only when the module itself can't be found, as for `import numpyy`. `NameError` is for using a name that was never defined in your own code, and `AttributeError` is what you'd get by looking the name up on the module, as in `math.sinn`, rather than importing it.


### Question 2

Running the script `summary.py` stops with this traceback:

```
Traceback (most recent call last):
  File "summary.py", line 13, in <module>
    load([1, 2, 3])
    ~~~~^^^^^^^^^^^
  File "summary.py", line 2, in load
    return summarise(values[1:])
  File "summary.py", line 6, in summarise
    return values[0] + pick(values, len(values))
                       ~~~~^^^^^^^^^^^^^^^^^^^^^
  File "summary.py", line 10, in pick
    return values[i]
           ~~~~~~^^^
IndexError: list index out of range
```

In which function was the `IndexError` raised?

- **(A)** `summarise`
- **(B)** `load`
- **(C)** None: it was raised at the top level of the script, on line 13
- **(D)** `pick`

**Answer: (D)**

A Python traceback lists the calls *most recent call last*: the first frame is where the program started (line 13, at the top level), each frame below it is a call made from the one above, and the last frame is where the exception was raised. Here that is `pick`, where `values[i]` is evaluated with `i` equal to `len(values)`, one past the end of the list. The *cause* is arguably in `summarise`, which passed `len(values)` as an index: the last frame is the best place to start looking, but the fix can be further up. `load`'s slice `values[1:]` can't raise an `IndexError` (a slice past the end of a list is just shorter or empty), and line 13 only starts the chain of calls.


### Question 3

`flake8` checks a module without running it. Each option below is a complete module, one line long. In which of them does `flake8` report a problem?

- **(A)** `total = cuont + 1`
- **(B)** `last = [1, 2, 3][3]`
- **(C)** `ratio = len("abc") / 0`
- **(D)** `shout = str.uper("abc")`

**Answer: (A)**

`flake8` reads the code without running it. It checks that every name used is defined and every import is used, and checks the style, but it never works out the value of an expression or which attributes an object has. `cuont` is never defined, so `flake8` reports `F821 undefined name 'cuont'`; `len` and `str` are built-in names, so they are fine. The other three fail only when they run: `len("abc") / 0` raises `ZeroDivisionError` and `[1, 2, 3][3]` raises `IndexError` once the values are computed, and `str.uper("abc")` raises `AttributeError` because `str` has no method `uper`, which `flake8` doesn't check. Static analysis complements tests rather than replacing them.


### Question 4

While debugging a fitting script, you run these lines to check the shapes:

```python
import numpy as np

y = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
y_fit = np.array([1.1, 1.9, 3.2, 3.9, 5.1])
residual = y - y_fit
```

What is `residual.shape`?

- **(A)** `(5, 5)`
- **(B)** None: the subtraction raises a `ValueError`
- **(C)** `(5,)`
- **(D)** `(5, 1)`

**Answer: (A)**

`y` is a column, with shape `(5, 1)`, and `y_fit` has shape `(5,)`. NumPy broadcasts the two to the common shape `(5, 5)`: every value of `y` minus every value of `y_fit`. No error is raised, so code that goes on to average `residual`, e.g. into an RMS error, silently gives a wrong answer. This is the bug in the lecture's MFE exercise, reduced to a few lines. To get one residual per point, make the shapes match: `y.ravel() - y_fit` has shape `(5,)`, and `y - y_fit[:, None]` has shape `(5, 1)`. NumPy raises a `ValueError` only when two shapes cannot be broadcast together at all.


### Question 5

In IPython you ran a script with `%run script.py`, and it stopped with a traceback. What do you type at the next IPython prompt to inspect the variables, as they were at that moment, in the function where the exception was raised?

- **(A)** `%tb`
- **(B)** `%debug`
- **(C)** `%xmode Verbose`
- **(D)** `%pdb on`

**Answer: (B)**

`%debug` starts the debugger *post mortem*: it opens in the frame where the last exception was raised, with the variables as they were, so you can print them (`p name`) and move up the call stack (`u`). `%tb` only prints the last traceback again. `%pdb on` and `%xmode Verbose` both change what happens to exceptions raised *later*: the first opens the debugger automatically, the second shows variables in the traceback. Either would mean running the script again.


### Question 6

You are debugging with `pdb`, stopped at the line

```python
denoised_img = local_mean(noisy_img, size=size)
```

Which command takes you *inside* `local_mean`, to its first line?

- **(A)** `c`
- **(B)** `d`
- **(C)** `n`
- **(D)** `s`

**Answer: (D)**

`s` (step) steps into the function called on the current line. `n` (next) runs the whole call and stops at the next line of the *current* function, so you never see inside `local_mean`. `c` (continue) runs on until the next breakpoint or the end of the program. `d` (down) moves down the call stack that already exists, towards where execution stopped, without running anything, so it can't enter a call that hasn't been made yet.


### Question 7

`test_shapes.py` is the only file in its folder:

```python
class TestSquare:
    def __init__(self):
        self.side = 2

    def test_area(self):
        assert self.side**2 == 4
```

What happens when you run `pytest` in that folder?

- **(A)** It runs `test_area`, which fails, because `self.side` was never set
- **(B)** It runs `test_area`, which passes
- **(C)** It collects no tests, and warns that it cannot collect `TestSquare`
- **(D)** It stops with an error, because a test class may not define `__init__`

**Answer: (C)**

`pytest` collects test methods from classes whose names start with `Test`, but only if the class has no `__init__` method. `TestSquare` defines `__init__`, so `pytest` skips it: it reports `collected 0 items` and a `PytestCollectionWarning` that it cannot collect `TestSquare` because it has an `__init__` constructor. The run doesn't stop with an error, and `test_area` never runs, so it neither passes nor fails. Easy to miss: nothing is tested, and nothing fails. Do any set-up inside the test itself instead.


### Question 8

Each of these assertions is run on its own. Which of them pass?

```python
import pytest

assert 0.1 + 0.2 == 0.3                     # (i)
assert 0.1 + 0.2 == pytest.approx(0.3)      # (ii)
assert 0.1 + 0.2 == pytest.approx(0.3001)   # (iii)
```

- **(A)** (ii) only
- **(B)** (i) and (ii)
- **(C)** (ii) and (iii)
- **(D)** All three

**Answer: (A)**

`0.1`, `0.2` and `0.3` can't be stored exactly in binary floating point, and `0.1 + 0.2` is `0.30000000000000004`, so the exact comparison (i) fails. `pytest.approx(0.3)` compares to within a relative tolerance of $10^{-6}$ by default, so (ii) passes. That tolerance is tight: `0.3001` differs from `0.3` by about 3 parts in $10^{4}$, far more than $10^{-6}$, so (iii) fails. `approx` allows for rounding error, not for a result that is merely close; a looser tolerance has to be chosen on purpose, e.g. `pytest.approx(0.3001, rel=1e-3)`. Lecture 7 explains where the rounding comes from.


### Question 9

Each option fills in `EXCEPTION` and `CALL` in this test file:

```python
import pytest


def test_conversion():
    with pytest.raises(EXCEPTION):
        CALL
```

For which option does the test **fail**?

- **(A)** `ValueError` and `int("three")`
- **(B)** `TypeError` and `int(None)`
- **(C)** `Exception` and `int("three")`
- **(D)** `ValueError` and `int("3")`

**Answer: (D)**

`pytest.raises(E)` passes if the code inside the `with` block raises `E` or a subclass of it. It fails if nothing is raised (`DID NOT RAISE`) or if an exception of another type escapes. `int("3")` simply returns 3, so nothing is raised and that test fails. The other three pass. `int("three")` raises a `ValueError`: the argument has the right type, a string, but a value that isn't a number. `int(None)` raises a `TypeError`, because `None` is the wrong type of argument. And `Exception` matches the `ValueError` from `int("three")`, because `ValueError` is a subclass of `Exception`. In a real test, name the specific exception: `pytest.raises(Exception)` would also pass if the code failed for some unrelated reason.


### Question 10

One of your tests fails. Which command reruns the tests and opens the debugger at the point where that test fails, so that you can inspect its variables?

- **(A)** `pytest --trace`
- **(B)** `pytest --debug`
- **(C)** `pytest --pdb`
- **(D)** `pytest -x`

**Answer: (C)**

`--pdb` drops into the debugger where a test fails, whether through a failed `assert` or an unexpected exception. `--trace` stops at the *start* of every test instead, so you would have to step through from the beginning. `-x` stops the whole run after the first failure, without a debugger. `--debug` writes a log of what `pytest` itself is doing (`pytestdebug.log`), for debugging `pytest`, not your code.


### Question 11

What does this print?

```python
import numpy as np

a = np.array([[1, 2], [3, 4]])
a[0], a[1] = a[1], a[0]
print(a.tolist())
```

- **(A)** Nothing: the line before `print` raises a `ValueError`
- **(B)** `[[3, 4], [3, 4]]`
- **(C)** `[[3, 4], [1, 2]]`
- **(D)** `[[1, 2], [1, 2]]`

**Answer: (B)**

Python first evaluates the right-hand side, the tuple `(a[1], a[0])`. For a NumPy array these are *views* of rows 1 and 0, not copies. The first assignment, `a[0] = a[1]`, copies `[3, 4]` into row 0, and the view of row 0 in the tuple now shows `[3, 4]` too. So the second assignment, which copies that view into row 1, copies `[3, 4]` again, and both rows end up `[3, 4]`. With a list of lists, the same line does swap the rows, because the tuple holds the two row objects themselves. This is the bug the `l4mod` exercise's tests find in `gauss`. To swap rows of an array, use `a[[0, 1]] = a[[1, 0]]` (fancy indexing makes a copy).


### Question 12

`names.py` contains

```python
def initials(name):
    """Initials of a first and last name.

    >>> initials("Ada Lovelace")
    EXPECTED
    """
    first, last = name.split()
    return first[0] + last[0]
```

With which line in place of `EXPECTED` does the doctest pass when you run `pytest --doctest-modules`?

- **(A)** `AL`
- **(B)** `'AL'`
- **(C)** `"AL"`

**Answer: (B)**

A doctest compares the *text* that the interactive interpreter would show with the expected output, character for character. For a value, the interpreter shows its `repr`, and the `repr` of a string is in single quotes: `'AL'`. `AL` is what `print` would show, not what the interpreter shows, and `"AL"` is the same string in different quotes. Both of those doctests fail, with `Expected` and `Got` lines showing the difference. The safe way to write a doctest is to run the call in the interpreter and copy what it shows.
